# CSE 291 / DSC 291 PA3 — Speculative Decoding

In this notebook you will implement and benchmark a single-sequence (batch=1) speculative decoder.

Recap of the algorithm:

1. A small **draft** model proposes `k` tokens autoregressively starting from the current context.
2. The large **target** model verifies the proposal in **one** forward pass (a single batched pass over the `L + k` length sequence).
3. Tokens are accepted greedily up to the first mismatch with the target's argmax. After the first mismatch, the target's own next token is appended and the loop restarts.

Default model pair (public weights, runs on any GPU with >=4 GB VRAM):

- target: `EleutherAI/pythia-1.4b-deduped`
- draft:  `EleutherAI/pythia-160m-deduped`

If you don't have GPU access, the same code paths run on CPU but you won't see a meaningful speedup.

## Setup

In [1]:
import os
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Speculative Decoder

In [2]:
class SpeculativeDecoder:
    def __init__(self, target_model_name: str, draft_model_name: str, device: str = "cuda"):
        """Initialize the speculative decoder with a target and a draft model."""
        self.device = device
        self.target_model, self.target_tokenizer = self.initialize_target_model(target_model_name)
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(draft_model_name)

        assert self.target_tokenizer.get_vocab() == self.draft_tokenizer.get_vocab(), (
            "Target and draft must share a vocabulary"
        )

    def _inference_dtype(self) -> torch.dtype:
        """Choose a dtype supported by the requested device."""
        device_type = torch.device(self.device).type
        if device_type == "cuda":
            return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        if device_type == "mps":
            return torch.float16
        return torch.float32

    def _synchronize(self) -> None:
        """Synchronize CUDA before reading wall-clock timings."""
        if torch.device(self.device).type == "cuda":
            torch.cuda.synchronize()

    @staticmethod
    def _append_tokens(input_ids: torch.Tensor, attention_mask: torch.Tensor,
                       token_ids: List[int]) -> Tuple[torch.Tensor, torch.Tensor]:
        """Append token IDs and matching attention-mask entries."""
        if not token_ids:
            return input_ids, attention_mask
        new_tokens = torch.tensor([token_ids], dtype=input_ids.dtype, device=input_ids.device)
        new_mask = torch.ones_like(new_tokens, dtype=attention_mask.dtype)
        return (
            torch.cat([input_ids, new_tokens], dim=1),
            torch.cat([attention_mask, new_mask], dim=1),
        )

    def _reset_target_cache(self) -> None:
        self._target_past_key_values = None
        self._target_cached_input_ids = None

    @staticmethod
    def _crop_past_key_values(past_key_values, max_length: int):
        """Crop either a modern Transformers Cache or a legacy tuple cache."""
        if past_key_values is None:
            return None
        if hasattr(past_key_values, "crop"):
            past_key_values.crop(max_length)
            return past_key_values
        return tuple(
            tuple(state[..., :max_length, :] for state in layer)
            for layer in past_key_values
        )

    def _can_reuse_target_cache(self, input_ids: torch.Tensor) -> bool:
        cached_ids = getattr(self, "_target_cached_input_ids", None)
        past_key_values = getattr(self, "_target_past_key_values", None)
        if cached_ids is None or past_key_values is None:
            return False
        cached_length = cached_ids.shape[1]
        return (
            cached_length < input_ids.shape[1]
            and torch.equal(cached_ids, input_ids[:, :cached_length])
        )

    def initialize_target_model(self, model_name: str):
        """Load the larger target model with caching enabled."""
        print(f"Loading target model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self._inference_dtype(),
        ).to(self.device)
        model.eval()
        model.config.use_cache = True
        model.config.pad_token_id = tokenizer.pad_token_id
        model.generation_config.pad_token_id = tokenizer.pad_token_id
        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        """Load the smaller draft model."""
        print(f"Loading draft model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self._inference_dtype(),
        ).to(self.device)
        model.eval()
        model.config.use_cache = True
        model.config.pad_token_id = tokenizer.pad_token_id
        model.generation_config.pad_token_id = tokenizer.pad_token_id
        return model, tokenizer

    def generate_draft_tokens(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                             num_speculative_tokens: int = 10) -> torch.Tensor:
        """
        Generate `num_speculative_tokens` draft tokens with the draft model.

        Args:
            input_ids: Input token IDs (tensor of shape [1, seq_len]).
            attention_mask: Corresponding attention mask.
            num_speculative_tokens: Number of tokens to speculate.

        Returns:
            Tensor of shape [1, num_speculative_tokens] containing the draft tokens.
        """
        if input_ids.shape[0] != 1:
            raise ValueError("This decoder supports batch size 1 only")
        if num_speculative_tokens < 0:
            raise ValueError("num_speculative_tokens must be non-negative")
        if num_speculative_tokens == 0:
            return input_ids[:, :0]

        generated_tokens = []
        model_input_ids = input_ids
        running_attention_mask = attention_mask
        past_key_values = None

        # The prefix is evaluated once. Subsequent draft steps reuse its KV cache.
        with torch.inference_mode():
            for _ in range(num_speculative_tokens):
                outputs = self.draft_model(
                    input_ids=model_input_ids,
                    attention_mask=running_attention_mask,
                    past_key_values=past_key_values,
                    use_cache=True,
                )
                next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                generated_tokens.append(next_token)
                past_key_values = outputs.past_key_values
                model_input_ids = next_token
                running_attention_mask = torch.cat(
                    [running_attention_mask, torch.ones_like(next_token, dtype=attention_mask.dtype)],
                    dim=1,
                )

        return torch.cat(generated_tokens, dim=1)

    def verify_tokens_vectorized(self, input_ids: torch.Tensor, draft_tokens: torch.Tensor,
                               attention_mask: torch.Tensor) -> Tuple[List[int], int]:
        """
        Vectorized verification: verify all draft tokens in one forward pass using the target model.

        Args:
            input_ids: The current input token IDs (shape [1, L]).
            draft_tokens: Draft tokens from the draft model (shape [1, k]).
            attention_mask: The current attention mask for input_ids.

        Returns:
            accepted_tokens: List of accepted token IDs.
            accepted_position: Index of the first rejected token (if all accepted, equals draft_tokens.shape[1]).
        """
        if input_ids.shape[0] != 1 or draft_tokens.shape[0] != 1:
            raise ValueError("This decoder supports batch size 1 only")
        num_draft_tokens = draft_tokens.shape[1]
        if num_draft_tokens == 0:
            self._next_target_token = None
            return [], 0

        candidate_ids = torch.cat([input_ids, draft_tokens], dim=1)
        candidate_mask = torch.cat(
            [attention_mask, torch.ones_like(draft_tokens, dtype=attention_mask.dtype)],
            dim=1,
        )

        cached_length = 0
        past_key_values = None
        if self._can_reuse_target_cache(input_ids):
            past_key_values = self._target_past_key_values
            cached_length = self._target_cached_input_ids.shape[1]

        with torch.inference_mode():
            outputs = self.target_model(
                input_ids=candidate_ids[:, cached_length:],
                attention_mask=candidate_mask,
                past_key_values=past_key_values,
                use_cache=True,
            )

        # Logit position L - 1 predicts the first draft token. The final logit
        # predicts either the correction token or the all-accepted bonus token.
        input_length = input_ids.shape[1]
        logit_start = input_length - 1 - cached_length
        target_tokens = outputs.logits[:, logit_start:logit_start + num_draft_tokens + 1, :].argmax(dim=-1)
        matches = target_tokens[:, :num_draft_tokens].eq(draft_tokens)[0]
        mismatch_positions = (~matches).nonzero(as_tuple=False)
        accepted_position = (
            int(mismatch_positions[0].item()) if mismatch_positions.numel() else num_draft_tokens
        )
        self._next_target_token = int(target_tokens[0, accepted_position].item())

        # Keep only the verified prefix. The correction/bonus token is processed
        # alongside the next proposal, so its KV state is not needed yet.
        valid_cache_length = input_length + accepted_position
        self._target_past_key_values = self._crop_past_key_values(
            outputs.past_key_values,
            valid_cache_length,
        )
        self._target_cached_input_ids = candidate_ids[:, :valid_cache_length].detach().clone()
        accepted_tokens = draft_tokens[0, :accepted_position].tolist()
        return accepted_tokens, accepted_position

    def speculative_decode(self, prompt: str, max_tokens: int = 100,
                          num_speculative_tokens: int = 4,
                          verbose: bool = True) -> str:
        """
        Main speculative decoding algorithm with vectorized verification.

        Args:
            prompt: Input text.
            max_tokens: Maximum number of tokens to generate (excluding prompt).
            num_speculative_tokens: Number of tokens to speculate per iteration.

        Returns:
            Generated text.
        """
        # Tokenize prompt
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        prompt_length = input_ids.shape[1]

        if max_tokens < 0:
            raise ValueError("max_tokens must be non-negative")
        if num_speculative_tokens <= 0:
            raise ValueError("num_speculative_tokens must be positive")

        # Initialize counters for performance tracking
        generated_tokens = 0
        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0
        eos_token_id = self.target_tokenizer.eos_token_id
        self._reset_target_cache()
        self._synchronize()
        start_time = time.time()

        while generated_tokens < max_tokens:
            proposal_length = min(num_speculative_tokens, max_tokens - generated_tokens)
            draft_tokens = self.generate_draft_tokens(
                input_ids,
                attention_mask,
                num_speculative_tokens=proposal_length,
            )
            accepted_tokens, _ = self.verify_tokens_vectorized(
                input_ids,
                draft_tokens,
                attention_mask,
            )
            total_draft_tokens_proposed += draft_tokens.shape[1]

            # Stop exactly at EOS even if later speculative tokens also matched.
            if eos_token_id in accepted_tokens:
                accepted_tokens = accepted_tokens[:accepted_tokens.index(eos_token_id) + 1]
            input_ids, attention_mask = self._append_tokens(input_ids, attention_mask, accepted_tokens)
            generated_tokens += len(accepted_tokens)
            total_draft_tokens_accepted += len(accepted_tokens)
            if (eos_token_id in accepted_tokens) or generated_tokens >= max_tokens:
                break

            # This token came from the same target forward pass used for verification.
            target_token = self._next_target_token
            input_ids, attention_mask = self._append_tokens(input_ids, attention_mask, [target_token])
            generated_tokens += 1
            if target_token == eos_token_id:
                break

        # Calculate performance metrics
        self._synchronize()
        elapsed_time = time.time() - start_time
        acceptance_rate = total_draft_tokens_accepted / total_draft_tokens_proposed if total_draft_tokens_proposed > 0 else 0
        self.last_decode_stats = {
            "elapsed_time": elapsed_time,
            "generated_tokens": generated_tokens,
            "tokens_per_second": generated_tokens / elapsed_time if elapsed_time else float("inf"),
            "draft_tokens_proposed": total_draft_tokens_proposed,
            "draft_tokens_accepted": total_draft_tokens_accepted,
            "acceptance_rate": acceptance_rate,
        }

        if verbose:
            print(f"Generated {generated_tokens} tokens in {elapsed_time:.2f} seconds")
            print(f"Tokens per second: {self.last_decode_stats['tokens_per_second']:.2f}")
            print(f"Draft token acceptance rate: {acceptance_rate:.2%}")

        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    def benchmark(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_runs: int = 3,
        num_speculative_tokens: int = 4,
        compare_baseline: bool = True,
        warmup: bool = True,
    ) -> Dict:
        results = {
            "speculative": {"times": [], "tokens_per_second": [], "acceptance_rates": []},
            "baseline": {"times": [], "tokens_per_second": []} if compare_baseline else None,
        }

        if warmup:
            print("Running one unmeasured warm-up iteration...")
            self.speculative_decode(
                prompt,
                max_tokens=max_tokens,
                num_speculative_tokens=num_speculative_tokens,
                verbose=False,
            )
            if compare_baseline:
                inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)
                with torch.inference_mode():
                    self.target_model.generate(
                        input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=max_tokens,
                        do_sample=False,
                        use_cache=True,
                        pad_token_id=self.target_tokenizer.pad_token_id,
                    )
                self._synchronize()

        for _ in range(num_runs):
            self.speculative_decode(
                prompt,
                max_tokens=max_tokens,
                num_speculative_tokens=num_speculative_tokens,
            )
            elapsed = self.last_decode_stats["elapsed_time"]
            output_tokens = self.last_decode_stats["generated_tokens"]
            results["speculative"]["times"].append(elapsed)
            results["speculative"]["tokens_per_second"].append(output_tokens / elapsed)
            results["speculative"]["acceptance_rates"].append(self.last_decode_stats["acceptance_rate"])

        if compare_baseline:
            for _ in range(num_runs):
                inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)
                self._synchronize()
                t0 = time.time()
                with torch.inference_mode():
                    output_ids = self.target_model.generate(
                        input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=max_tokens,
                        do_sample=False,
                        use_cache=True,
                        pad_token_id=self.target_tokenizer.pad_token_id,
                    )
                self._synchronize()
                elapsed = time.time() - t0
                output_tokens = output_ids.shape[1] - input_ids.shape[1]
                results["baseline"]["times"].append(elapsed)
                results["baseline"]["tokens_per_second"].append(output_tokens / elapsed)

        for method in results:
            if results[method] is not None:
                results[method]["avg_time"] = sum(results[method]["times"]) / num_runs
                results[method]["avg_tokens_per_second"] = (
                    sum(results[method]["tokens_per_second"]) / num_runs
                )
        results["speculative"]["avg_acceptance_rate"] = (
            sum(results["speculative"]["acceptance_rates"]) / num_runs
        )
        if compare_baseline:
            results["speedup"] = (
                results["baseline"]["avg_time"] / results["speculative"]["avg_time"]
            )
            results["latency_reduction"] = (
                1 - results["speculative"]["avg_time"] / results["baseline"]["avg_time"]
            ) * 100
        return results

## Test

In [3]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"
draft_model_name = "EleutherAI/pythia-160m-deduped"

decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# Use the same prompt for each k so the acceptance-rate and speedup comparison is fair.
benchmark_prompt = "The future of artificial intelligence is"
sweep_results = {}
for num_speculative_tokens in [2, 4, 8, 16]:
    print(f"\nBenchmarking k={num_speculative_tokens}: {benchmark_prompt}")
    results = decoder.benchmark(
        prompt=benchmark_prompt,
        max_tokens=100,
        num_runs=3,
        num_speculative_tokens=num_speculative_tokens,
        compare_baseline=True,
    )
    sweep_results[num_speculative_tokens] = results
    print(f"  Speculative: {results['speculative']['avg_time']:.2f}s, "
          f"{results['speculative']['avg_tokens_per_second']:.2f} tok/s")
    print(f"  Baseline:    {results['baseline']['avg_time']:.2f}s, "
          f"{results['baseline']['avg_tokens_per_second']:.2f} tok/s")
    print(f"  Acceptance rate: {results['speculative']['avg_acceptance_rate']:.2%}")
    print(f"  Speedup: {results['speedup']:.2f}x  |  Latency reduction: {results['latency_reduction']:.2f}%")

print("\nMarkdown table for part3/report.md:")
print("| k | Acceptance rate | Speculative tok/s | Baseline tok/s | Speedup |")
print("|---:|---:|---:|---:|---:|")
for k, results in sweep_results.items():
    print(f"| {k} | {results['speculative']['avg_acceptance_rate']:.2%} | "
          f"{results['speculative']['avg_tokens_per_second']:.2f} | "
          f"{results['baseline']['avg_tokens_per_second']:.2f} | "
          f"{results['speedup']:.2f}x |")

Loading target model: EleutherAI/pythia-1.4b-deduped


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Benchmarking k=2: The future of artificial intelligence is
Running one unmeasured warm-up iteration...
Generated 100 tokens in 1.35 seconds
Tokens per second: 74.11
Draft token acceptance rate: 91.55%
Generated 100 tokens in 1.35 seconds
Tokens per second: 74.29
Draft token acceptance rate: 91.55%
Generated 100 tokens in 1.33 seconds
Tokens per second: 75.22
Draft token acceptance rate: 91.55%
  Speculative: 1.34s, 74.54 tok/s
  Baseline:    1.69s, 59.11 tok/s
  Acceptance rate: 91.55%
  Speedup: 1.26x  |  Latency reduction: 20.70%

Benchmarking k=4: The future of artificial intelligence is
Running one unmeasured warm-up iteration...
Generated 100 tokens in 1.57 seconds
Tokens per second: 63.83
Draft token acceptance rate: 81.05%
Generated 100 tokens in 1.34 seconds
Tokens per second: 74.68
Draft token acceptance rate: 81.05%
Generated 100 tokens in 1.29 seconds
Tokens per second: 77.28
Draft token acceptance rate: 81.05%
  Speculative: 1.40s, 71.93 tok/s
  Baseline:    1.71s, 58.43 t

## Bonus 3.B — Tree speculation or n-gram lookup decoding (10 pts)

Implement one stronger speculative-decoding variant and benchmark it
against the baseline:

- **Tree / multi-branch speculation** (Medusa / EAGLE-2 style): verify
  several candidate continuations in a single target forward pass.
- **N-gram lookup decoding** (Prompt Lookup Decoding): draft the next
  tokens from an n-gram cache built over the running sequence instead of
  (or in addition to) the draft model.

Re-run the benchmark with your bonus decoder and report the speedup and
acceptance rate in your write-up. See the bonus rubric in `../README.md`.

In [4]:
# Bonus implementation goes here.
# Re-run the benchmark above with your bonus decoder and copy the numbers into
# your report.

In [5]:
class PromptLookupDecoder(SpeculativeDecoder):
    """Hybrid prompt-lookup decoder with draft-model fallback."""

    @classmethod
    def from_speculative_decoder(cls, decoder: SpeculativeDecoder):
        """Reuse already-loaded models instead of allocating a second copy."""
        lookup_decoder = cls.__new__(cls)
        lookup_decoder.device = decoder.device
        lookup_decoder.target_model = decoder.target_model
        lookup_decoder.target_tokenizer = decoder.target_tokenizer
        lookup_decoder.draft_model = decoder.draft_model
        lookup_decoder.draft_tokenizer = decoder.draft_tokenizer
        return lookup_decoder

    def generate_lookup_tokens(self, input_ids: torch.Tensor,
                               max_lookup_tokens: int = 8,
                               min_ngram_size: int = 2,
                               max_ngram_size: int = 4) -> torch.Tensor:
        """Copy a continuation following the longest repeated suffix n-gram."""
        if input_ids.shape[0] != 1:
            raise ValueError("This decoder supports batch size 1 only")
        if max_lookup_tokens < 0:
            raise ValueError("max_lookup_tokens must be non-negative")
        if min_ngram_size <= 0 or max_ngram_size < min_ngram_size:
            raise ValueError("Expected 0 < min_ngram_size <= max_ngram_size")
        if max_lookup_tokens == 0:
            return input_ids[:, :0]

        tokens = input_ids[0].tolist()
        largest_ngram = min(max_ngram_size, len(tokens) - 1)
        for ngram_size in range(largest_ngram, min_ngram_size - 1, -1):
            suffix = tokens[-ngram_size:]
            # Prefer the latest previous match when multiple continuations exist.
            for start in range(len(tokens) - ngram_size - 1, -1, -1):
                if tokens[start:start + ngram_size] != suffix:
                    continue
                continuation = tokens[
                    start + ngram_size:start + ngram_size + max_lookup_tokens
                ]
                if continuation:
                    return torch.tensor(
                        [continuation],
                        dtype=input_ids.dtype,
                        device=input_ids.device,
                    )
        return input_ids[:, :0]

    def prompt_lookup_decode(self, prompt: str, max_tokens: int = 100,
                             max_lookup_tokens: int = 8,
                             fallback_draft_tokens: int = 2,
                             min_ngram_size: int = 2,
                             max_ngram_size: int = 4,
                             verbose: bool = True) -> str:
        """Decode greedily using free n-gram proposals when available."""
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        if max_tokens < 0:
            raise ValueError("max_tokens must be non-negative")
        if max_lookup_tokens <= 0 or fallback_draft_tokens <= 0:
            raise ValueError("Proposal lengths must be positive")

        generated_tokens = 0
        total_tokens_proposed = 0
        total_tokens_accepted = 0
        lookup_attempts = 0
        lookup_matches = 0
        lookup_tokens_proposed = 0
        lookup_tokens_accepted = 0
        fallback_tokens_proposed = 0
        fallback_tokens_accepted = 0
        eos_token_id = self.target_tokenizer.eos_token_id
        self._reset_target_cache()
        self._synchronize()
        start_time = time.time()

        while generated_tokens < max_tokens:
            remaining_tokens = max_tokens - generated_tokens
            lookup_attempts += 1
            proposal = self.generate_lookup_tokens(
                input_ids,
                max_lookup_tokens=min(max_lookup_tokens, remaining_tokens),
                min_ngram_size=min_ngram_size,
                max_ngram_size=max_ngram_size,
            )
            proposal_source = "lookup"
            if proposal.shape[1] == 0:
                proposal_source = "draft_model"
                proposal = self.generate_draft_tokens(
                    input_ids,
                    attention_mask,
                    num_speculative_tokens=min(fallback_draft_tokens, remaining_tokens),
                )
            else:
                lookup_matches += 1

            accepted_tokens, _ = self.verify_tokens_vectorized(
                input_ids,
                proposal,
                attention_mask,
            )
            total_tokens_proposed += proposal.shape[1]
            if proposal_source == "lookup":
                lookup_tokens_proposed += proposal.shape[1]
            else:
                fallback_tokens_proposed += proposal.shape[1]

            if eos_token_id in accepted_tokens:
                accepted_tokens = accepted_tokens[:accepted_tokens.index(eos_token_id) + 1]
            input_ids, attention_mask = self._append_tokens(input_ids, attention_mask, accepted_tokens)
            generated_tokens += len(accepted_tokens)
            total_tokens_accepted += len(accepted_tokens)
            if proposal_source == "lookup":
                lookup_tokens_accepted += len(accepted_tokens)
            else:
                fallback_tokens_accepted += len(accepted_tokens)
            if (eos_token_id in accepted_tokens) or generated_tokens >= max_tokens:
                break

            target_token = self._next_target_token
            input_ids, attention_mask = self._append_tokens(input_ids, attention_mask, [target_token])
            generated_tokens += 1
            if target_token == eos_token_id:
                break

        self._synchronize()
        elapsed_time = time.time() - start_time
        self.last_lookup_stats = {
            "elapsed_time": elapsed_time,
            "generated_tokens": generated_tokens,
            "tokens_per_second": generated_tokens / elapsed_time if elapsed_time else float("inf"),
            "tokens_proposed": total_tokens_proposed,
            "tokens_accepted": total_tokens_accepted,
            "acceptance_rate": total_tokens_accepted / total_tokens_proposed if total_tokens_proposed else 0,
            "lookup_attempts": lookup_attempts,
            "lookup_matches": lookup_matches,
            "lookup_match_rate": lookup_matches / lookup_attempts if lookup_attempts else 0,
            "lookup_tokens_proposed": lookup_tokens_proposed,
            "lookup_tokens_accepted": lookup_tokens_accepted,
            "lookup_acceptance_rate": lookup_tokens_accepted / lookup_tokens_proposed if lookup_tokens_proposed else 0,
            "fallback_tokens_proposed": fallback_tokens_proposed,
            "fallback_tokens_accepted": fallback_tokens_accepted,
        }

        if verbose:
            print(f"Generated {generated_tokens} tokens in {elapsed_time:.2f} seconds")
            print(f"Tokens per second: {self.last_lookup_stats['tokens_per_second']:.2f}")
            print(f"Overall proposal acceptance rate: {self.last_lookup_stats['acceptance_rate']:.2%}")
            print(f"Lookup match rate: {self.last_lookup_stats['lookup_match_rate']:.2%}")
            print(f"Lookup-token acceptance rate: {self.last_lookup_stats['lookup_acceptance_rate']:.2%}")
        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    def benchmark_lookup(self, prompt: str, max_tokens: int = 100,
                         num_runs: int = 3, max_lookup_tokens: int = 8,
                         fallback_draft_tokens: int = 2,
                         min_ngram_size: int = 2,
                         max_ngram_size: int = 4,
                         warmup: bool = True) -> Dict:
        """Benchmark hybrid prompt lookup against target-only decoding."""
        results = {
            "lookup": {"times": [], "tokens_per_second": [], "acceptance_rates": [],
                       "lookup_match_rates": [], "lookup_acceptance_rates": []},
            "baseline": {"times": [], "tokens_per_second": []},
        }

        if warmup:
            print("Running one unmeasured prompt-lookup warm-up iteration...")
            self.prompt_lookup_decode(
                prompt, max_tokens=max_tokens, max_lookup_tokens=max_lookup_tokens,
                fallback_draft_tokens=fallback_draft_tokens,
                min_ngram_size=min_ngram_size, max_ngram_size=max_ngram_size,
                verbose=False,
            )
            inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
            with torch.inference_mode():
                self.target_model.generate(
                    inputs["input_ids"].to(self.device),
                    attention_mask=inputs["attention_mask"].to(self.device),
                    max_new_tokens=max_tokens, do_sample=False, use_cache=True,
                    pad_token_id=self.target_tokenizer.pad_token_id,
                )
            self._synchronize()

        for _ in range(num_runs):
            self.prompt_lookup_decode(
                prompt, max_tokens=max_tokens, max_lookup_tokens=max_lookup_tokens,
                fallback_draft_tokens=fallback_draft_tokens,
                min_ngram_size=min_ngram_size, max_ngram_size=max_ngram_size,
            )
            stats = self.last_lookup_stats
            results["lookup"]["times"].append(stats["elapsed_time"])
            results["lookup"]["tokens_per_second"].append(stats["tokens_per_second"])
            results["lookup"]["acceptance_rates"].append(stats["acceptance_rate"])
            results["lookup"]["lookup_match_rates"].append(stats["lookup_match_rate"])
            results["lookup"]["lookup_acceptance_rates"].append(stats["lookup_acceptance_rate"])

        for _ in range(num_runs):
            inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
            input_ids = inputs["input_ids"].to(self.device)
            attention_mask = inputs["attention_mask"].to(self.device)
            self._synchronize()
            t0 = time.time()
            with torch.inference_mode():
                output_ids = self.target_model.generate(
                    input_ids, attention_mask=attention_mask, max_new_tokens=max_tokens,
                    do_sample=False, use_cache=True,
                    pad_token_id=self.target_tokenizer.pad_token_id,
                )
            self._synchronize()
            elapsed_time = time.time() - t0
            generated_tokens = output_ids.shape[1] - input_ids.shape[1]
            results["baseline"]["times"].append(elapsed_time)
            results["baseline"]["tokens_per_second"].append(generated_tokens / elapsed_time)

        for method in ["lookup", "baseline"]:
            results[method]["avg_time"] = sum(results[method]["times"]) / num_runs
            results[method]["avg_tokens_per_second"] = sum(results[method]["tokens_per_second"]) / num_runs
        results["lookup"]["avg_acceptance_rate"] = sum(results["lookup"]["acceptance_rates"]) / num_runs
        results["lookup"]["avg_lookup_match_rate"] = sum(results["lookup"]["lookup_match_rates"]) / num_runs
        results["lookup"]["avg_lookup_acceptance_rate"] = sum(results["lookup"]["lookup_acceptance_rates"]) / num_runs
        results["speedup"] = results["baseline"]["avg_time"] / results["lookup"]["avg_time"]
        return results


lookup_decoder = PromptLookupDecoder.from_speculative_decoder(decoder)

# A repeated prompt exposes the intended prompt-lookup use case. The fallback
# draft model still handles iterations where the running sequence has no match.
lookup_prompt = (
    "Speculative decoding can reduce generation latency. "
    "Speculative decoding can reduce generation latency. "
    "Speculative decoding can"
)
lookup_sweep_results = {}
for max_lookup_tokens in [2, 4, 8, 16]:
    print(f"\nBenchmarking prompt lookup with max_lookup_tokens={max_lookup_tokens}")
    results = lookup_decoder.benchmark_lookup(
        prompt=lookup_prompt,
        max_tokens=100,
        num_runs=3,
        max_lookup_tokens=max_lookup_tokens,
        fallback_draft_tokens=2,
        min_ngram_size=2,
        max_ngram_size=4,
    )
    lookup_sweep_results[max_lookup_tokens] = results
    print(f"  Prompt lookup: {results['lookup']['avg_time']:.2f}s, "
          f"{results['lookup']['avg_tokens_per_second']:.2f} tok/s")
    print(f"  Baseline:      {results['baseline']['avg_time']:.2f}s, "
          f"{results['baseline']['avg_tokens_per_second']:.2f} tok/s")
    print(f"  Overall acceptance rate: {results['lookup']['avg_acceptance_rate']:.2%}")
    print(f"  Lookup match rate: {results['lookup']['avg_lookup_match_rate']:.2%}")
    print(f"  Lookup-token acceptance rate: {results['lookup']['avg_lookup_acceptance_rate']:.2%}")
    print(f"  Speedup: {results['speedup']:.2f}x")

print("\nMarkdown table for the bonus section of part3/report.md:")
print("| Max lookup tokens | Overall acceptance | Lookup match rate | Lookup-token acceptance | Speedup |")
print("|---:|---:|---:|---:|---:|")
for max_lookup_tokens, results in lookup_sweep_results.items():
    print(f"| {max_lookup_tokens} | {results['lookup']['avg_acceptance_rate']:.2%} | "
          f"{results['lookup']['avg_lookup_match_rate']:.2%} | "
          f"{results['lookup']['avg_lookup_acceptance_rate']:.2%} | "
          f"{results['speedup']:.2f}x |")


Benchmarking prompt lookup with max_lookup_tokens=2
Running one unmeasured prompt-lookup warm-up iteration...
Generated 100 tokens in 0.65 seconds
Tokens per second: 152.72
Overall proposal acceptance rate: 100.00%
Lookup match rate: 100.00%
Lookup-token acceptance rate: 100.00%
Generated 100 tokens in 0.66 seconds
Tokens per second: 152.43
Overall proposal acceptance rate: 100.00%
Lookup match rate: 100.00%
Lookup-token acceptance rate: 100.00%
Generated 100 tokens in 0.66 seconds
Tokens per second: 152.48
Overall proposal acceptance rate: 100.00%
Lookup match rate: 100.00%
Lookup-token acceptance rate: 100.00%
  Prompt lookup: 0.66s, 152.54 tok/s
  Baseline:      1.71s, 58.37 tok/s
  Overall acceptance rate: 100.00%
  Lookup match rate: 100.00%
  Lookup-token acceptance rate: 100.00%
  Speedup: 2.61x

Benchmarking prompt lookup with max_lookup_tokens=4
Running one unmeasured prompt-lookup warm-up iteration...
Generated 100 tokens in 0.39 seconds
Tokens per second: 258.23
Overall pro